# Data Loading

## Dataset Description

Your challenge in this competition is to identify which species (birds, amphibians, mammals, reptiles, insects) are calling in recordings made in the Brazilian Pantanal. This is an important task for scientists who monitor animal populations for conservation purposes. More accurate solutions could enable more comprehensive monitoring.

This competition uses a hidden test set. When your submitted notebook is scored, the actual test data will be made available to your notebook.

### Files

`train_audio/` The training data consists of short recordings of individual bird, amphibian, reptile, mammal, and insect sounds generously uploaded by users of xeno-canto.org and iNaturalist. These files have been resampled to 32 kHz where applicable to match the test set audio and converted to the ogg format. Filenames consist of [collection][file_id_in_collection].ogg. The training data should have nearly all relevant files; we expect there is no benefit to looking for more on xeno-canto.org or iNaturalist and appreciate your cooperation in limiting the burden on their servers. If you do, please make sure to adhere to the scraping rules of these data portals.

`test_soundscapes/` When you submit a notebook, the test_soundscapes directory will be populated with approximately 600 recordings to be used for scoring. They are 1 minute long and in ogg audio format, resampled to 32 kHz. The file names have the general form of BC2026_Test_<file ID>_<site>_<date>_<time in UTC>.ogg (e.g., file BC2026_Test_0001_S05_20250227_010002.ogg has file ID 0001, was recorded at site S05 on Feb 27 2025 at 01:00 UTC). It should take your submission notebook approximately five minutes to load all the test soundscapes. Not all species from the training data actually occur in the test data.

`train_soundscapes/` Additional audio data from roughly the same recording locations as the test_soundscapes. Filenames follow the same naming convention as the test_soundscapes; although some recording sites overlap between train and test, precise recording dates and times do NOT overlap with recordings of the hidden test data. This year, some of the train_soundscapes have been labeled by expert annotators, and we provide the ground truth for a subset of train_soundscapes in train_soundscapes_labels.csv with columns filename referencing the soundscape file, start and end referencing the 5-second segment for which column primary_label provides a semicolon-separated list of species codes that have been marked as present in this segment.

**Important note:** Some species with occurrences in the hidden test data might only have train samples in the labeled portion of train_soundscapes and not in the train_audio (XC and iNat data). However, not all species from train_soundscapes have occurrences in the test_soundscapes.

`train.csv` A wide range of metadata is provided for the training data. The most directly relevant fields are:

- primary_label: A code for the species (eBird code for birds, iNaturalist taxon ID for non-birds). You can review detailed information about the species by appending codes to eBird and iNaturalist taxon URL, such as https://ebird.org/species/brnowl for the Barn Owl or https://www.inaturalist.org/taxa/41970 for the Jaguar. Not all species have their own pages; some links might fail.

- secondary_labels: List of species labels that have been marked by recordists to also occur in the recording. Can be incomplete.

- latitude & longitude: Coordinates for where the recording was taken. Some bird species may have local call 'dialects,' so you may want to seek geographic diversity in your training data.

- author: The user who provided the recording. Unknown if no name was provided.

- filename: The name of the associated audio file.

- rating: Values in 1..5 (1 - low quality, 5 - high quality; 0.5 reduction in rating when background species are present) provided by users of Xeno-canto; 0 implies no rating is available; iNaturalist does not provide quality ratings.

- collection: Either XC or iNat, indicating which collection the recording was taken from. Filenames also reference the collection and the ID within that collection.

`sample_submission.csv` A valid sample submission.

- row_id: A slug of [soundscape_filename]_[end_time] for the prediction; e.g., Segment 00:15-00:20 of 1-minute test soundscape BC2026_Test_0001_S05_20250227_010002.ogg has row ID BC2026_Test_0001_S05_20250227_010002_20.

- [species_id]: There are 234 species ID columns. You will need to predict the probability of the presence of each species for each row.
taxonomy.csv - Data on the different species, including iNaturalist taxon ID and class name (Aves, Amphibia, Mammalia, Insecta, Reptilia). Most insect species in this competition have not been identified on species level and instead occur as sonotypes (e.g., 47158son16 as insect sonotype 16); these sonotypes are treated as classes despite the lack of species ID and some of them also occur in the test data. The 234 rows of this file represent the 234 class columns in the submission file. primary_label specifies the submission file column name.

`recording_location.txt` - Some high-level information on the recording location (Pantanal, Brazil).

In [1]:
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df_train = pd.read_csv("../birdclef-2026/train.csv")
df_train_soundscapes = pd.read_csv("../birdclef-2026/train_soundscapes_labels.csv")
df_taxonomy = pd.read_csv("../birdclef-2026/taxonomy.csv")

In [3]:
df_train

,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35544,yeofly1,[],['call'],-6.7281,-76.4181,Tolmomyias sulphurescens,Yellow-olive Flatbill,Aves,16567,GABRIEL LEITE,by-nc-sa,4.0,https://xeno-canto.org/762473/download,yeofly1/XC762473.ogg,XC
35545,yeofly1,[],"['call', ' song']",-16.0538,-49.6040,Tolmomyias sulphurescens,Yellow-olive Flatbill,Aves,16567,JAYRSON ARAUJO DE OLIVEIRA,by-nc-sa,5.0,https://xeno-canto.org/818328/download,yeofly1/XC818328.ogg,XC
35546,yeofly1,[],['song'],5.9002,-74.8485,Tolmomyias sulphurescens,Yellow-olive Flatbill,Aves,16567,Jerome Fischer,by-nc-sa,4.0,https://xeno-canto.org/425545/download,yeofly1/XC425545.ogg,XC
35547,yeofly1,[],['song'],3.3821,-61.4464,Tolmomyias sulphurescens,Yellow-olive Flatbill,Aves,16567,GABRIEL LEITE,by-nc-sa,4.0,https://xeno-canto.org/456230/download,yeofly1/XC456230.ogg,XC


In [4]:
df_train_soundscapes

,filename,start,end,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380
3,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:15,00:00:20,22961;23158;24321;517063;65380
4,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:20,00:00:25,22961;23158;24321;517063;65380
...,...,...,...,...
1473,BC2026_Train_0055_S22_20220219_200000.ogg,00:00:35,00:00:40,555146;65380
1474,BC2026_Train_0055_S22_20220219_200000.ogg,00:00:40,00:00:45,517063;555146
1475,BC2026_Train_0055_S22_20220219_200000.ogg,00:00:45,00:00:50,517063;555146
1476,BC2026_Train_0055_S22_20220219_200000.ogg,00:00:50,00:00:55,517063;555146;66971


In [5]:
df_taxonomy

,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia
...,...,...,...,...,...
229,yebela1,16714,Elaenia flavogaster,Yellow-bellied Elaenia,Aves
230,yecmac,73272,Primolius auricollis,Golden-collared Macaw,Aves
231,yecpar,19215,Brotogeris chiriri,Yellow-chevroned Parakeet,Aves
232,yehcar1,1432779,Daptrius chimachima,Yellow-headed Caracara,Aves


For now I am going to ignore the secondary labels as they concern only a small fraction of the data.

In [6]:
print(
    f"Percentage of train audio with secondary labels: {(df_train['secondary_labels'] != '[]').sum() / len(df_train) * 100:.2f}%"
)

Percentage of train audio with secondary labels: 12.30%


There are very many different types of calls, songs, etc... It would likely make sense combine them into different categories e.g. songs, calls, etc so that there can be clearer similarities between the calories. I've listed them here.

In [7]:
import ast

list_types = df_train["type"].unique().tolist()
list_types = [ast.literal_eval(lst) for lst in list_types]
possible_types = set(x for sublist in list_types for x in sublist)
print(list_types)
print(possible_types)

[[], ['uncertain'], ['advertisement call'], ['territorial call'], ['flight call'], ['advertisement call', ' chorus'], ['reproductive call ?'], ['alarm call'], ['call'], ['mating call'], ['social call'], ['advertisement call', ' territorial call'], ['song'], ['call', ' mechanical sound'], ['mechanical sound'], ['sniffing'], ['call', ' song'], ['call', ' scolding'], ['canto'], ['call', ' subsong'], ['alarm call', ' llamadas de contacto'], ['voz de contacto de adulto.'], ['song', ' llamado de adulto.'], ['voz de contacto de pichón.'], ['duet', ' song'], ['duet'], ['alarm call', ' call'], ['aberrant'], ['dueto'], ['duet', ' song', ' song in duet'], ['song', ' duo'], ['fast low call'], ['chamado'], ['dawn song'], ['call', ' duet'], ['call', ' wing sound'], ['alarm call', ' call', ' juvenile call'], ['song', ' long song'], ['song', ' complex song'], ['song', " 'laughing' song"], ['song', " 'barking' song"], ['song', ' simple song'], ['call', ' dawn song', ' duet'], ['call', ' duet', ' song']

In [8]:
print(
    f"Percentage of train audio with type: {(df_train['type'] != '[]').sum() / len(df_train) * 100:.2f}%"
)

Percentage of train audio with type: 63.50%


For now I will just use a `LabelEncoder` from Scikit-learn to encode the primary label. I fit the encoder on `df_taxonomy` since there are some animals that are only in either `train_audio` or `train_soundscapes`.

In [9]:
from sklearn.preprocessing import MultiLabelBinarizer

le_classes = MultiLabelBinarizer()
le_primary = MultiLabelBinarizer()

le_classes.fit(df_taxonomy["class_name"].apply(lambda x: [x]))
le_primary.fit(df_taxonomy["primary_label"].apply(lambda x: [x]))

print(f"Number of Classes: {len(le_classes.classes_)}")
print(f"Number of Species: {len(le_primary.classes_)}")

Number of Classes: 5
Number of Species: 234


In [10]:
df_taxonomy

,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia
...,...,...,...,...,...
229,yebela1,16714,Elaenia flavogaster,Yellow-bellied Elaenia,Aves
230,yecmac,73272,Primolius auricollis,Golden-collared Macaw,Aves
231,yecpar,19215,Brotogeris chiriri,Yellow-chevroned Parakeet,Aves
232,yehcar1,1432779,Daptrius chimachima,Yellow-headed Caracara,Aves


Since we only have primary labels and not class names in `df_train_soundscapes` we have a mapping from primary label to class name.

In [11]:
primary_to_class = {}
for primary_label, class_name in zip(
    df_taxonomy["primary_label"], df_taxonomy["class_name"]
):
    primary_to_class[primary_label] = class_name

In [12]:
from datetime import datetime


def time_to_seconds(t):
    dt = datetime.strptime(t, "%H:%M:%S")
    return 3600 * dt.hour + 60 * dt.minute + dt.second


x_train_files = df_train[["filename"]]
x_train_soundscapes_files = df_train_soundscapes[["filename", "start", "end"]]
x_train_soundscapes_files["start"] = x_train_soundscapes_files["start"].apply(
    time_to_seconds
)
x_train_soundscapes_files["end"] = x_train_soundscapes_files["end"].apply(
    time_to_seconds
)
x_train_soundscapes_files

,filename,start,end
0,BC2026_Train_0039_S22_20211231_201500.ogg,0,5
1,BC2026_Train_0039_S22_20211231_201500.ogg,5,10
2,BC2026_Train_0039_S22_20211231_201500.ogg,10,15
3,BC2026_Train_0039_S22_20211231_201500.ogg,15,20
4,BC2026_Train_0039_S22_20211231_201500.ogg,20,25
...,...,...,...
1473,BC2026_Train_0055_S22_20220219_200000.ogg,35,40
1474,BC2026_Train_0055_S22_20220219_200000.ogg,40,45
1475,BC2026_Train_0055_S22_20220219_200000.ogg,45,50
1476,BC2026_Train_0055_S22_20220219_200000.ogg,50,55


In [13]:
y_train_classes = le_classes.transform(df_train["class_name"].apply(lambda x: [x]))
print(y_train_classes.shape)

y_train_primary = le_primary.transform(df_train["primary_label"].apply(lambda x: [x]))
print(y_train_primary.shape)

(35549, 5)
(35549, 234)


In [14]:
y_soundscapes_classes = le_classes.transform(
    df_train_soundscapes["primary_label"]
    .apply(lambda x: x.split(";"))
    .apply(lambda x: list({primary_to_class[primary_label] for primary_label in x}))
)
print(y_soundscapes_classes.shape)

y_soundscapes_primary = le_primary.transform(
    df_train_soundscapes["primary_label"].apply(lambda x: x.split(";"))
)
print(y_soundscapes_primary.shape)

(1478, 5)
(1478, 234)


Our functions for analyzing the sounds should probably take as input a row from either `x_train` or `x_train_soundscapes`.

In [15]:
x_train_soundscapes_files.iloc[:10]

,filename,start,end
0,BC2026_Train_0039_S22_20211231_201500.ogg,0,5
1,BC2026_Train_0039_S22_20211231_201500.ogg,5,10
2,BC2026_Train_0039_S22_20211231_201500.ogg,10,15
3,BC2026_Train_0039_S22_20211231_201500.ogg,15,20
4,BC2026_Train_0039_S22_20211231_201500.ogg,20,25
5,BC2026_Train_0039_S22_20211231_201500.ogg,25,30
6,BC2026_Train_0039_S22_20211231_201500.ogg,30,35
7,BC2026_Train_0039_S22_20211231_201500.ogg,35,40
8,BC2026_Train_0039_S22_20211231_201500.ogg,40,45
9,BC2026_Train_0039_S22_20211231_201500.ogg,45,50


The example function below is how I think it can be good to write functions for finding features from the data.

In [16]:
# example function for getting features
def example_analysis(x):
    if x.shape == (1,):  # Audio
        audio, sr = librosa.load(
            f"../birdclef-2026/train_audio/{x['filename']}", sr=32000
        )
    elif x.shape == (3,):  # Soundscape
        audio, sr = librosa.load(
            f"../birdclef-2026/train_soundscapes/{x['filename']}",
            sr=32000,
            offset=x["start"],
            duration=x["end"] - x["start"],
        )
    else:
        raise ValueError

    # plt.title(f"{x['filename']}")
    S = np.abs(librosa.stft(audio))
    S_db = librosa.amplitude_to_db(S)
    # librosa.display.specshow(S_db)
    # plt.show()

    return pd.Series({"spec_min": S_db.min(), "spec_max": S_db.max()})

Takes about ~20 mins to load the whole dataset (can probably be improved)

In [ ]:
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", module="librosa")

features = pd.DataFrame(
    [
        example_analysis(row)
        for _, row in tqdm(x_train_files.iterrows(), total=len(x_train_files))
    ]
)